# Test 3 Option C: Speech-to-Speech (InferenceState official API)
Prerequisite: Run moshiko2.0.ipynb first. Uses moshi's officially supported InferenceState pipeline.

In [ ]:
# Step 1: Set up InferenceState
import torch, time, os
from moshi.models import loaders
from moshi.run_inference import InferenceState

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1
CFG_COEF = 1.0

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Disable bfloat16 autocast for T4
torch.set_autocast_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("Autocast and TF32 disabled.")

# Build CheckpointInfo from local files
Q8_DIR = FOLDERS["models_q8"]
mimi_path = os.path.join(Q8_DIR, "tokenizer-e351c8d8-checkpoint125.safetensors")
moshi_path = os.path.join(Q8_DIR, "model.q8.safetensors")

# Create CheckpointInfo pointing to our local files
checkpoint_info = loaders.CheckpointInfo(
    hf_repo="kyutai/moshiko-pytorch-q8",
    moshi_weight=moshi_path,
    mimi_weight=mimi_path,
    tokenizer=None,  # Will use tokenizer from the repo
    config=None,     # Will use config from the repo
)

# Load Mimi
print("\nLoading Mimi...")
start = time.time()
mimi = checkpoint_info.get_mimi(device=DEVICE)
mimi.float()  # Force float32 for T4
mimi.eval()
print(f"  Mimi loaded in {time.time()-start:.1f}s")

# Load text tokenizer
print("\nLoading text tokenizer...")
text_tokenizer = checkpoint_info.get_text_tokenizer()
print("  Text tokenizer loaded.")

# Load Moshi LM (use float16 instead of bfloat16 for T4)
print("\nLoading Moshiko LM (float16 for T4)...")
start = time.time()
lm = checkpoint_info.get_moshi(device=DEVICE, dtype=torch.float16)
lm.eval()
print(f"  Moshiko LM loaded in {time.time()-start:.1f}s")

# Create InferenceState
print("\nCreating InferenceState...")
inference_state = InferenceState(
    checkpoint_info,
    mimi=mimi,
    text_tokenizer=text_tokenizer,
    lm=lm,
    batch_size=BATCH_SIZE,
    cfg_coef=CFG_COEF,
    device=DEVICE,
    temp=0.8,
    temp_text=0.7,
)
print("  InferenceState ready.")

torch.cuda.empty_cache()
if DEVICE == "cuda":
    print(f"  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Store globally
import builtins
builtins.mimi = mimi
builtins.inference_state = inference_state
builtins.DEVICE = DEVICE

print("\nReady for inference.")

In [ ]:
# Step 2: Run speech-to-speech inference
import torch, soundfile as sf, sphn
import numpy as np
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000

# Input audio path
input_path = f"{FOLDERS['audio_in']}/test_input_440hz.wav"
if not os.path.exists(input_path):
    print(f"Input file not found: {input_path}")
    print("Run Test 1 first, or upload a speech WAV and update input_path.")
else:
    # Load audio with sphn (InferenceState expects this format)
    print(f"Loading: {input_path}")
    in_pcm, sr = sphn.read(input_path, sample_rate=SAMPLE_RATE)
    # in_pcm shape: [channels, samples] -> convert to [1, 1, samples]
    in_pcm = torch.from_numpy(in_pcm).float().to(DEVICE)
    if in_pcm.dim() == 1:
        in_pcm = in_pcm.unsqueeze(0).unsqueeze(0)
    elif in_pcm.dim() == 2:
        in_pcm = in_pcm[0:1].unsqueeze(0)  # Take first channel, add batch dim

    print(f"  Duration: {in_pcm.shape[-1]/SAMPLE_RATE:.2f}s")
    print(f"  Shape: {in_pcm.shape}")

    # Run inference
    print("\nRunning inference...")
    gen_start = time.time()

    with torch.no_grad():
        out_items = inference_state.run(in_pcm)

    gen_time = time.time() - gen_start
    print(f"Inference completed in {gen_time:.1f}s")

    if out_items:
        for idx, (text_tokens, audio_tokens) in enumerate(out_items):
            print(f"\n  Output {idx}:")
            print(f"    Text tokens: {text_tokens.shape}")
            print(f"    Audio tokens: {audio_tokens.shape}")

            # Decode audio tokens to waveform
            with torch.no_grad():
                # audio_tokens shape: [B, K, T] - need to pass through mimi.decode
                response_audio = mimi.decode(audio_tokens)

            # Save output
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_path = f"{FOLDERS['audio_out']}/test3c_response_{ts}.wav"
            sf.write(output_path, response_audio.squeeze().cpu().numpy(), SAMPLE_RATE)

            duration = response_audio.shape[-1] / SAMPLE_RATE
            print(f"    Response duration: {duration:.2f}s")
            print(f"    Output saved: {output_path}")

            # Decode text tokens
            text_tokenizer = inference_state.text_tokenizer
            decoded_text = ""
            for tid in text_tokens.flatten().tolist():
                if tid not in [0, 3]:  # Skip special tokens
                    piece = text_tokenizer.id_to_piece(tid)
                    decoded_text += piece.replace("\u2581", " ")
            if decoded_text.strip():
                print(f"    Transcribed text: {decoded_text.strip()}")

            # Play audio
            print("\n  Input audio:")
            wav_np, _ = sf.read(input_path)
            display(Audio(wav_np, rate=SAMPLE_RATE))
            print("  Response audio:")
            display(Audio(response_audio.squeeze().cpu().numpy(), rate=SAMPLE_RATE))

            # Save results
            result = {
                "test": "Speech-to-speech (Option C: InferenceState)",
                "timestamp": ts,
                "input_file": input_path,
                "output_file": output_path,
                "input_duration_s": round(in_pcm.shape[-1]/SAMPLE_RATE, 2),
                "response_duration_s": round(duration, 2),
                "gen_time_s": round(gen_time, 2),
                "text_tokens_count": text_tokens.numel(),
                "audio_frames": audio_tokens.shape[-1],
                "decoded_text": decoded_text.strip(),
            }
            import json
            with open(f"{FOLDERS['outputs']}/test3c_results_{ts}.json", "w") as f:
                json.dump(result, f, indent=2)

            print(f"\nTest 3 Option C complete. Files saved to Drive.")
    else:
        print("No output generated. Check model configuration.")